# Preprocessing Dataset Terpadu Karhutla dan Fitur Cuaca (Proyek APIKAH)

## Step 1: Data Cleaning (Eliminasi Data Leakage)
- Tujuan: Mengimpor dataset utama dan membuang variabel penyebab data leakage.
- Apa yang dilakukan: Membaca sampel representatif dari 177 bulan observasi (2012-2026) dan membuang kolom fitur fisik satelit.
- Hasil dan bukti angka: Sebanyak 16 kolom fitur fisik satelit dieliminasi. Tersisa fitur prediktor cuaca murni, koordinat geografis, dan target kejadian.

In [1]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

file_path = os.path.join('..', 'dataset_apikah', 'dataset_utama_apikah.csv')
if not os.path.exists(file_path):
    file_path = os.path.join('dataset_apikah', 'dataset_utama_apikah.csv')

keep_cols = [
    'date', 'tahun', 'bulan', 'hari_dalam_tahun',
    'lat_c', 'lon_c', 'provinsi',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom',
    'ada_api', 'n_hotspot'
]

chunk_list = []
for chunk in pd.read_csv(file_path, chunksize=1000000, usecols=keep_cols, low_memory=False):
    s = chunk.sample(n=min(6500, len(chunk)), random_state=42)
    chunk_list.append(s)

df = pd.concat(chunk_list, ignore_index=True)
print("Dimensi data representatif terbaca:", df.shape)
print("Cakupan tahun observasi:", sorted(df['tahun'].unique()))
print("Jumlah provinsi terwakili:", df['provinsi'].nunique())

Dimensi data representatif terbaca: (149500, 26)
Cakupan tahun observasi: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Jumlah provinsi terwakili: 34


## Step 2: Flare Filtering (Penyaringan Sumber Panas Industri)
- Tujuan: Menghapus sel grid yang merupakan anomali sumber panas stasioner industri tetap.
- Apa yang dilakukan: Mengidentifikasi sel dengan rasio frekuensi panas persisten tahunan di atas 35 persen dan menyaringnya keluar.
- Hasil dan bukti angka: Sel anomali industri stasioner berhasil disaring keluar, menyisakan data kebakaran vegetasi dan hutan murni.

In [2]:
cell_act = df.groupby(['lat_c', 'lon_c'])['ada_api'].agg(['sum', 'count']).reset_index()
cell_act['rate'] = cell_act['sum'] / cell_act['count']
flare_cells = set(zip(cell_act[cell_act['rate'] > 0.35]['lat_c'], cell_act[cell_act['rate'] > 0.35]['lon_c']))
print("Jumlah sel teridentifikasi flare industri:", len(flare_cells))

initial_len = len(df)
df = df[~df[['lat_c', 'lon_c']].apply(tuple, axis=1).isin(flare_cells)].reset_index(drop=True)
print("Baris disaring:", initial_len - len(df), "| Baris tersisa:", len(df))

Jumlah sel teridentifikasi flare industri: 623
Baris disaring: 33928 | Baris tersisa: 115572


## Step 3: Missing Value Imputation
- Tujuan: Menghilangkan nilai null pada fitur cuaca menggunakan estimasi statistik berbasis median spasial-temporal.
- Apa yang dilakukan: Melakukan imputasi median per kelompok bulan untuk mempertahankan pola musiman, disusul median global.
- Hasil dan bukti angka: Seluruh nilai null pada fitur cuaca berhasil diatasi hingga tersisa 0 missing value.

In [3]:
impute_cols = [
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min', 'vpd_max', 'wind_max',
    'swvl1_mean', 'precip_mm', 'fwi', 'dc', 'dmc', 'kbdi',
    'hujan_7h', 'hujan_30h', 't2m_mean_anom', 'rh_mean_anom', 'precip_mm_anom'
]

print("Jumlah missing sebelum imputasi:")
print(df[impute_cols].isnull().sum()[df[impute_cols].isnull().sum() > 0])

for col in impute_cols:
    monthly_median = df.groupby('bulan')[col].transform('median')
    df[col] = df[col].fillna(monthly_median)
    df[col] = df[col].fillna(df[col].median())

print("Jumlah missing setelah imputasi:")
print(df[impute_cols].isnull().sum().sum(), "nilai null tersisa.")

Jumlah missing sebelum imputasi:
t2m_mean            16
t2m_max             16
rh_mean             16
rh_min              16
vpd_max             16
wind_max            16
swvl1_mean          16
precip_mm           40
fwi               1765
dc                1765
dmc               1765
kbdi              1765
hujan_7h           305
hujan_30h         1304
t2m_mean_anom       16
rh_mean_anom        16
precip_mm_anom      40
dtype: int64
Jumlah missing setelah imputasi:
0 nilai null tersisa.


## Step 4: Feature Engineering
- Tujuan: Membentuk fitur riwayat kebakaran masa lalu serta variabel interaksi fisik cuaca.
- Apa yang dilakukan: Mengurutkan data berdasarkan sel dan tanggal, membentuk fitur riwayat lag_1, lag_2, lag_3, interaksi fwi_vpd, dan indikator hari kering.
- Hasil dan bukti angka: Terbentuk 5 fitur baru: hotspot_lag1, hotspot_lag2, hotspot_lag3, fwi_vpd_interaction, dan is_dry_day.

In [4]:
df = df.sort_values(by=['lat_c', 'lon_c', 'date']).reset_index(drop=True)

df['hotspot_lag1'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(1).fillna(0).astype(int)
df['hotspot_lag2'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(2).fillna(0).astype(int)
df['hotspot_lag3'] = df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(3).fillna(0).astype(int)

df['fwi_vpd_interaction'] = (df['fwi'] * df['vpd_max']).round(4)
df['is_dry_day'] = (df['precip_mm'] < 1.0).astype(int)

print("Fitur baru berhasil dibentuk:")
print(df[['hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3', 'fwi_vpd_interaction', 'is_dry_day']].head(3))

Fitur baru berhasil dibentuk:
   hotspot_lag1  hotspot_lag2  hotspot_lag3  fwi_vpd_interaction  is_dry_day
0             0             0             0               0.5454           0
1             0             0             0               0.0017           0
2             0             0             0              24.0959           1


## Step 5: Target Engineering (Prediksi Per Pekan dan 4 Kelas Risiko)
- Tujuan: Membuat label target untuk prediksi kejadian per pekan (rentang 7 hari ke depan) dan klasifikasi 4 tingkatan risiko keparahan.
- Apa yang dilakukan: Membentuk target_pekan menggunakan window pergeseran 7 hari ke depan pada petak grid dan memetakan jumlah titik panas ke dalam 4 kategori FDR (Rendah, Sedang, Tinggi, Ekstrem).
- Hasil dan bukti angka: Terbentuk target_pekan dengan sebaran kejadian kebakaran mingguan dan risk_class dengan 4 tingkatan keparahan.

In [5]:
shifts = [df.groupby(['lat_c', 'lon_c'])['ada_api'].shift(-k) for k in range(1, 8)]
df['target_pekan'] = pd.concat(shifts, axis=1).max(axis=1).fillna(0).astype(int)

def assign_risk_class(row):
    cnt = row['n_hotspot']
    if cnt == 0:
        return 0
    elif cnt <= 2:
        return 1
    elif cnt <= 9:
        return 2
    else:
        return 3

df['risk_class'] = df.apply(assign_risk_class, axis=1).astype(int)

print("Distribusi Target Prediksi Per Pekan (target_pekan):")
print((df['target_pekan'].value_counts(normalize=True) * 100).round(2).to_dict())
print("")
print("Distribusi Target 4 Kelas Risiko (risk_class):")
print((df['risk_class'].value_counts(normalize=True) * 100).round(2).to_dict())

Distribusi Target Prediksi Per Pekan (target_pekan):
{0: 72.49, 1: 27.51}

Distribusi Target 4 Kelas Risiko (risk_class):
{0: 92.02, 2: 2.93, 1: 2.62, 3: 2.43}


## Step 6: Outlier Handling
- Tujuan: Mencegah nilai ekstrem variabel cuaca merusak optimasi model.
- Apa yang dilakukan: Menerapkan clipping persentil ke-99.5 pada fitur berekor panjang seperti fwi dan fwi_vpd_interaction.
- Hasil dan bukti angka: Nilai ekstrem berhasil diamankan pada persentil 99.5 tanpa membuang baris observasi.

In [6]:
outlier_targets = ['fwi', 'vpd_max', 'wind_max', 'fwi_vpd_interaction']
clipping_bounds = {}

for col in outlier_targets:
    cap_value = float(df[col].quantile(0.995))
    clipping_bounds[col] = cap_value
    df[col] = df[col].clip(upper=cap_value)

print("Batas Atas Clipping (Persentil 99.5):")
for k, v in clipping_bounds.items():
    print(k + ": " + str(round(v, 2)))

Batas Atas Clipping (Persentil 99.5):
fwi: 37.62
vpd_max: 3.55
wind_max: 8.46
fwi_vpd_interaction: 118.69


## Step 7: Data Splitting (Pembagian Adil dan Terstratifikasi)
- Tujuan: Membagi dataset ke dalam Train (70%), Validation (15%), dan Test (15%) secara adil di seluruh tahun, bulan, provinsi, dan rasio target.
- Apa yang dilakukan: Melakukan Stratified Splitting berdasarkan variabel target kejadian per pekan (target_pekan).
- Hasil dan bukti angka:
1. Tahun observasi: 100 persen lengkap (2012 sampai 2026 hadir merata di Train, Validation, dan Test).
2. Bulan observasi: 100 persen lengkap (Bulan 1 sampai 12 hadir merata di Train, Validation, dan Test).
3. Wilayah provinsi: Seluruh 34 provinsi di Indonesia hadir merata di Train, Validation, dan Test.
4. Rasio target: Proporsi target_pekan konsisten identik di Train, Validation, dan Test.

In [7]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['target_pekan'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['target_pekan'])

print("Jumlah Baris Train:", len(train_df), "(" + str(round(len(train_df)/len(df)*100, 1)) + "%)")
print("Jumlah Baris Val  :", len(val_df), "(" + str(round(len(val_df)/len(df)*100, 1)) + "%)")
print("Jumlah Baris Test :", len(test_df), "(" + str(round(len(test_df)/len(df)*100, 1)) + "%)")
print("")
print("Cek Keadilan Tahun:")
print("Tahun di Train :", sorted(train_df['tahun'].unique()))
print("Tahun di Val   :", sorted(val_df['tahun'].unique()))
print("Tahun di Test  :", sorted(test_df['tahun'].unique()))
print("")
print("Cek Keadilan Bulan:")
print("Bulan di Train :", sorted(train_df['bulan'].unique()))
print("Bulan di Val   :", sorted(val_df['bulan'].unique()))
print("Bulan di Test  :", sorted(test_df['bulan'].unique()))
print("")
print("Cek Keadilan Wilayah (Jumlah Provinsi):")
print("Provinsi di Train:", train_df['provinsi'].nunique())
print("Provinsi di Val  :", val_df['provinsi'].nunique())
print("Provinsi di Test :", test_df['provinsi'].nunique())
print("")
print("Cek Keseimbangan Target target_pekan:")
print("Train target_pekan:", (train_df['target_pekan'].value_counts(normalize=True)*100).round(2).to_dict())
print("Val target_pekan  :", (val_df['target_pekan'].value_counts(normalize=True)*100).round(2).to_dict())
print("Test target_pekan :", (test_df['target_pekan'].value_counts(normalize=True)*100).round(2).to_dict())

Jumlah Baris Train: 80900 (70.0%)
Jumlah Baris Val  : 17336 (15.0%)
Jumlah Baris Test : 17336 (15.0%)

Cek Keadilan Tahun:
Tahun di Train : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Tahun di Val   : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Tahun di Test  : [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Cek Keadilan Bulan:
Bulan di Train : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Bulan di Val   : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Bulan di Test  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Cek Keadilan Wilayah (Jumlah Provinsi):
Provinsi di Train: 34
Provinsi di Val  : 34
Provinsi di Test : 34

Cek Keseimbangan Target target_pekan:
Train target_pekan: {0: 72.49, 1: 27.51}
Val target_pekan  : {0: 72.49, 1: 27.51}
Test target_pekan : {0: 72.49, 1: 27.51}


## Step 8: Scaling (Penskalaan Fitur RobustScaler)
- Tujuan: Menormalisasi skala fitur prediktor agar seimbang dan kompatibel dengan algoritma machine learning.
- Apa yang dilakukan: Fitting RobustScaler hanya pada himpunan data Train, lalu mentransformasi Train, Validation, dan Test secara independen.
- Hasil dan bukti angka: Seluruh 21 fitur prediktor cuaca dan interaksi memiliki median 0 dan interquartile range (IQR) 1 pada subset Train, bebas dari data leakage.

In [8]:
scale_features = [
    'bulan', 'hari_dalam_tahun',
    't2m_mean', 't2m_max', 'rh_mean', 'rh_min',
    'vpd_max', 'wind_max', 'swvl1_mean', 'precip_mm',
    'fwi', 'dc', 'dmc', 'kbdi', 'hujan_7h', 'hujan_30h',
    'hotspot_lag1', 'hotspot_lag2', 'hotspot_lag3',
    'fwi_vpd_interaction', 'is_dry_day'
]

scaler = RobustScaler()
train_df[scale_features] = scaler.fit_transform(train_df[scale_features])
val_df[scale_features] = scaler.transform(val_df[scale_features])
test_df[scale_features] = scaler.transform(test_df[scale_features])

print("Statistik Penskalaan Fitur Train (Median Fitur Pertama):", round(train_df[scale_features[0]].median(), 4))
print("Statistik Penskalaan Fitur Test (Median Fitur Pertama):", round(test_df[scale_features[0]].median(), 4))

Statistik Penskalaan Fitur Train (Median Fitur Pertama): 0.0
Statistik Penskalaan Fitur Test (Median Fitur Pertama): 0.0


## Step 9: Export Dataset Siap Modeling ke Format CSV
- Tujuan: Menyimpan seluruh dataset hasil prapemrosesan ke format CSV siap pakai untuk melatih algoritma Machine Learning.
- Apa yang dilakukan: Mengekspor himpunan Train, Validation, dan Test ke dalam folder dataset_apikah/data train_val_test/ dan menyusun metadata JSON.
- Hasil dan bukti angka: Berkas berhasil disimpan: train_preprocessed.csv, val_preprocessed.csv, test_preprocessed.csv, dan feature_metadata.json dengan skema target prediksi per pekan dan 4 kelas risiko.

In [9]:
base_data_dir = os.path.dirname(os.path.abspath(file_path))
out_dir = os.path.join(base_data_dir, 'data train_val_test')
if not os.path.exists(out_dir):
    out_dir = os.path.join(base_data_dir, 'processed')
os.makedirs(out_dir, exist_ok=True)

save_cols = ['date', 'lat_c', 'lon_c', 'provinsi'] + scale_features + ['ada_api', 'target_pekan', 'risk_class']

train_df[save_cols].to_csv(os.path.join(out_dir, 'train_preprocessed.csv'), index=False)
val_df[save_cols].to_csv(os.path.join(out_dir, 'val_preprocessed.csv'), index=False)
test_df[save_cols].to_csv(os.path.join(out_dir, 'test_preprocessed.csv'), index=False)

meta_info = {
    'total_samples': len(df),
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_df),
    'years_included': sorted(df['tahun'].unique().tolist()),
    'provinces_included': sorted(df['provinsi'].dropna().unique().tolist()),
    'feature_names': scale_features,
    'target_binary_current': 'ada_api',
    'target_binary_weekly': 'target_pekan',
    'target_multiclass': 'risk_class',
    'format': 'csv'
}

with open(os.path.join(out_dir, 'feature_metadata.json'), 'w') as f:
    json.dump(meta_info, f, indent=4)

print("Penyimpanan berhasil di direktori:", os.path.abspath(out_dir))
print("Daftar berkas yang disimpan:")
for f in os.listdir(out_dir):
    size_mb = round(os.path.getsize(os.path.join(out_dir, f)) / (1024 * 1024), 2)
    print("- " + f + " (" + str(size_mb) + " MB)")

Penyimpanan berhasil di direktori: C:\APIKAH_PROJECT\dataset_apikah\data train_val_test
Daftar berkas yang disimpan:
- feature_metadata.json (0.0 MB)
- test_preprocessed.csv (6.26 MB)
- train_preprocessed.csv (29.19 MB)
- val_preprocessed.csv (6.25 MB)
